# Bronze Layer — Data Ingestion Overview

Surveys every table ingested into the **bronze_lakehouse** so you can confirm what landed and inspect it.

| | |
|---|---|
| **Lakehouse** | bronze_lakehouse |
| **AOI** | Maple Ridge, BC — 20 km radius |
| **Layers** | Planetary Computer STAC, PC collections, DataBC WFS geology |

Run top-to-bottom. The default lakehouse must be attached (`bronze_lakehouse`).

## 1. Discover all bronze tables

In [ ]:
# List every table in the default lakehouse and keep the bronze_ ones
all_tables = [t.name for t in spark.catalog.listTables()]
bronze_tables = sorted([t for t in all_tables if t.startswith("bronze_")])

print(f"Found {len(bronze_tables)} bronze tables:\n")
for t in bronze_tables:
    print(f"  - {t}")

## 2. Row counts per table

In [ ]:
# Build a summary of row counts and column counts for each bronze table
from pyspark.sql import Row

summary = []
for t in bronze_tables:
    try:
        df = spark.table(t)
        summary.append(Row(table=t, rows=df.count(), columns=len(df.columns)))
    except Exception as e:
        summary.append(Row(table=t, rows=-1, columns=-1))
        print(f"  ! {t}: {e}")

summary_df = spark.createDataFrame(summary).orderBy("table")
print(f"Total bronze tables: {len(summary)}")
print(f"Total rows across all bronze tables: {sum(r.rows for r in summary if r.rows > 0)}")
display(summary_df)

## 3. Schema + sample rows for each table

In [ ]:
# Inspect schema and a few sample rows for every bronze table
for t in bronze_tables:
    print("=" * 80)
    print(f"TABLE: {t}")
    print("=" * 80)
    try:
        df = spark.table(t)
        print(f"Rows: {df.count()}  |  Columns: {len(df.columns)}")
        df.printSchema()
        display(df.limit(5))
    except Exception as e:
        print(f"  ! Could not read {t}: {e}")
    print()

## 4. Maps — one per layer, with real raster + clickable details

This renders a **separate map for each bronze layer** so overlapping rasters
don't pile up into mush:

- **STAC raster tables** (Sentinel-2, Sentinel-1, DEM, land cover, biomass…) →
  the **actual raster** is drawn as live tiles. The metadata bronze tables only
  store item footprints + asset names, so each item is re-fetched from the
  Planetary Computer data API to pull its `tilejson` tile layer (falling back to
  the `rendered_preview` PNG overlay). Each item's footprint is also drawn as a
  clickable outline.
- **DataBC geology tables** (quaternary, bedrock, faults) → drawn as their
  actual **GeoJSON geometries**.
- **Click any feature** (raster footprint or geology polygon) to see **every
  column** for that record in a scrollable popup.
- Red marker + circle = AOI center and 20 km radius. Use the layer control
  (top-right) to toggle individual items on/off.

> Raster tiles are fetched live from `planetarycomputer.microsoft.com`, so the
> Spark host needs outbound internet. A few item fetches per table keeps it fast.


In [ ]:
# Map helpers: ensure folium, fetch real raster tiles from Planetary Computer,
# and build a "show every column" popup.
import sys, subprocess, json, requests

try:
    import folium
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "folium", "-q"], check=True)
    import folium

LAT, LON, RADIUS_KM = 49.2193, -122.5984, 20
PC_STAC = "https://planetarycomputer.microsoft.com/api/stac/v1"
PC_ATTR = "Imagery &copy; Microsoft Planetary Computer"

bbox_cols = {"bbox_minx", "bbox_miny", "bbox_maxx", "bbox_maxy"}

# Distinct CSS-safe color per table (valid for Leaflet paths)
palette = [
    "blue", "green", "purple", "orange", "darkred", "teal", "darkgreen",
    "navy", "magenta", "brown", "gray", "steelblue", "crimson", "olive",
]
color_for = {t: palette[i % len(palette)] for i, t in enumerate(bronze_tables)}

# A few items per raster table keeps the live tile fetch fast.
MAX_RASTER_ITEMS = 5
MAX_VECTOR_FEATURES = 1000


def fetch_item(collection, item_id, timeout=30):
    """Fetch a STAC item from Planetary Computer. PC injects `tilejson` and
    `rendered_preview` assets that render the item with its collection's
    default config — no signing needed to read them."""
    url = f"{PC_STAC}/collections/{collection}/items/{item_id}"
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()


def tile_url_for_item(item, timeout=30):
    """Resolve an XYZ {z}/{x}/{y} raster tile template for a STAC item."""
    tj = (item.get("assets", {}) or {}).get("tilejson")
    if not tj or not tj.get("href"):
        return None
    try:
        doc = requests.get(tj["href"], timeout=timeout).json()
        tiles = doc.get("tiles") or []
        return tiles[0] if tiles else None
    except Exception:
        return None


def preview_url_for_item(item):
    """Fallback static PNG preview (covers the item footprint)."""
    a = (item.get("assets", {}) or {}).get("rendered_preview")
    return a.get("href") if a else None


def _esc(v):
    s = "" if v is None else str(v)
    if len(s) > 600:
        s = s[:600] + " \u2026"
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def popup_html(d, title=None):
    """Scrollable HTML table showing every field in a row dict."""
    rows = "".join(
        f"<tr><td style='font-weight:600;padding:2px 6px;vertical-align:top;"
        f"white-space:nowrap'>{_esc(k)}</td>"
        f"<td style='padding:2px 6px;word-break:break-word'>{_esc(v)}</td></tr>"
        for k, v in d.items()
    )
    head = (f"<div style='font-weight:700;margin-bottom:4px'>{_esc(title)}</div>"
            if title else "")
    return ("<div style='max-height:320px;max-width:440px;overflow:auto;"
            "font-family:system-ui,sans-serif;font-size:12px'>"
            + head + "<table>" + rows + "</table></div>")


print("Map helpers ready. Raster items per table:", MAX_RASTER_ITEMS)


In [ ]:
# One map per bronze layer: real raster tiles for STAC tables, GeoJSON for
# geology tables, every-column popups on click.

def new_map():
    m = folium.Map(location=[LAT, LON], zoom_start=11, tiles="CartoDB positron")
    folium.Marker(
        [LAT, LON], tooltip="AOI center — Maple Ridge, BC",
        icon=folium.Icon(color="red", icon="star"),
    ).add_to(m)
    folium.Circle(
        [LAT, LON], radius=RADIUS_KM * 1000, color="red",
        fill=False, weight=2, tooltip=f"{RADIUS_KM} km AOI",
    ).add_to(m)
    return m


for t in bronze_tables:
    color = color_for[t]
    try:
        df = spark.table(t)
    except Exception as e:
        print(f"  ! skip {t}: {e}")
        continue
    colset = set(df.columns)
    m = new_map()
    rendered = 0

    if bbox_cols.issubset(colset):
        # STAC raster table — draw the actual raster per item + clickable footprint
        rows = df.limit(MAX_RASTER_ITEMS).collect()
        for r in rows:
            d = r.asDict()
            minx, miny = d.get("bbox_minx"), d.get("bbox_miny")
            maxx, maxy = d.get("bbox_maxx"), d.get("bbox_maxy")
            item_id, collection = d.get("item_id"), d.get("collection")

            tile_url = preview = None
            if item_id and collection:
                try:
                    item = fetch_item(collection, item_id)
                    tile_url = tile_url_for_item(item)
                    preview = preview_url_for_item(item)
                except Exception as e:
                    print(f"    ! {t} {item_id}: {e.__class__.__name__}")

            if tile_url:
                folium.TileLayer(
                    tiles=tile_url, attr=PC_ATTR, name=str(item_id),
                    overlay=True, control=True, opacity=0.9,
                ).add_to(m)
                rendered += 1
            elif preview and None not in (minx, miny, maxx, maxy):
                folium.raster_layers.ImageOverlay(
                    image=preview, bounds=[[miny, minx], [maxy, maxx]],
                    opacity=0.9, name=str(item_id),
                ).add_to(m)
                rendered += 1

            if None not in (minx, miny, maxx, maxy):
                folium.Rectangle(
                    bounds=[[miny, minx], [maxy, maxx]],
                    color=color, weight=2, fill=False,
                    popup=folium.Popup(popup_html(d, title=item_id), max_width=460),
                    tooltip=str(item_id or t),
                ).add_to(m)
        label = f"{rendered} raster item(s)"

    elif "geometry_json" in colset:
        # Geology table — draw real GeoJSON geometries with full-detail popups
        rows = df.limit(MAX_VECTOR_FEATURES).collect()
        for r in rows:
            d = r.asDict()
            gj = d.get("geometry_json")
            if not gj:
                continue
            try:
                geom = json.loads(gj)
            except Exception:
                continue
            attrs = {k: v for k, v in d.items() if k != "geometry_json"}
            folium.GeoJson(
                geom,
                style_function=lambda x, c=color: {
                    "color": c, "weight": 2, "fillColor": c, "fillOpacity": 0.25,
                },
                popup=folium.Popup(popup_html(attrs, title=t), max_width=460),
                tooltip=t,
            ).add_to(m)
            rendered += 1
        label = f"{rendered} geometry feature(s)"

    else:
        print(f"  - {t}: no spatial columns (bbox_* or geometry_json) — skipped")
        continue

    folium.LayerControl(collapsed=False).add_to(m)
    print(f"  + {t}: {label} ({color})")

    header = (f"<h3 style='font-family:system-ui,sans-serif;margin:14px 0 4px'>"
              f"{t} <span style='font-weight:400;color:#666;font-size:13px'>"
              f"— {label}</span></h3>")
    try:
        displayHTML(header + m._repr_html_())
    except NameError:
        display(m)
